# Bone Cancer Detection Training Workflow

This notebook reproduces the project workflow used for:
1. augmenting the original bone dataset,
2. training ResNet-50 on the original dataset,
3. training ResNet-50 on the augmented dataset,
4. comparing validation checkpoints,
5. running external BTXRD prediction with the best model.

## How to run on another device
1. Clone or download the project folder.
2. Open this notebook in Jupyter Lab or VS Code.
3. Make sure the working directory is the project root.
4. Create a virtual environment and install dependencies.
5. Run the cells in order.

## Folder expectation
The project should contain:
- bone_dataset/
- bone_dataset_augmented/
- train_resnet50.py
- augment_dataset.py
- predict_external.py
- model checkpoints (.pt files)

## Recommended command for setup
```powershell
python -m venv .venv
.\.venv\Scripts\activate
pip install torch torchvision pillow
```


In [ ]:
from pathlib import Path
import os

project_root = Path(r"D:/BONE PROJECT BY AP/bone-cancer-detection-repo/bone-cancer-detection")
if project_root.exists():
    os.chdir(project_root)
    print(f"Working directory set to: {project_root}")
else:
    print("Project folder not found. Update project_root to match your machine.")


In [ ]:
import subprocess
from pathlib import Path

project_root = Path.cwd()
venv_python = project_root / '.venv' / 'Scripts' / 'python.exe'
print('Project root:', project_root)
print('Python executable exists:', venv_python.exists())
if not venv_python.exists():
    raise FileNotFoundError(
        'Python virtual environment not found. Create it first with: python -m venv .venv'
    )


In [ ]:
import subprocess
from pathlib import Path

project_root = Path.cwd()
augment_script = project_root / 'augment_dataset.py'
command = [
    str(project_root / '.venv' / 'Scripts' / 'python.exe'),
    str(augment_script),
    '--input-dir', 'bone_dataset',
    '--output-dir', 'bone_dataset_augmented',
    '--copies', '2',
    '--seed', '42',
]
print('Running augmentation...')
subprocess.run(command, cwd=str(project_root), check=True)
print('Augmentation complete.')


In [ ]:
import subprocess
from pathlib import Path

project_root = Path.cwd()
train_script = project_root / 'train_resnet50.py'
command = [
    str(project_root / '.venv' / 'Scripts' / 'python.exe'),
    str(train_script),
    '--data-dir', 'bone_dataset',
    '--epochs', '10',
    '--batch-size', '16',
    '--output', 'resnet50_bone_cancer.pt',
]
print('Training on original dataset...')
subprocess.run(command, cwd=str(project_root), check=True)
print('Training on original dataset complete.')


In [ ]:
import subprocess
from pathlib import Path

project_root = Path.cwd()
train_script = project_root / 'train_resnet50.py'
command = [
    str(project_root / '.venv' / 'Scripts' / 'python.exe'),
    str(train_script),
    '--data-dir', 'bone_dataset_augmented',
    '--epochs', '20',
    '--batch-size', '20',
    '--output', 'resnet50_bone_cancer_augmented_20ep_20bs.pt',
]
print('Training on augmented dataset...')
subprocess.run(command, cwd=str(project_root), check=True)
print('Training on augmented dataset complete.')


In [ ]:
import glob
import torch

checkpoint_rows = []
for path in sorted(glob.glob('*.pt')):
    try:
        ckpt = torch.load(path, map_location='cpu', weights_only=False)
        if isinstance(ckpt, dict) and 'best_valid_accuracy' in ckpt:
            checkpoint_rows.append((path, float(ckpt['best_valid_accuracy'])))
    except Exception:
        pass

if not checkpoint_rows:
    print('No checkpoints found.')
else:
    print('Saved checkpoint accuracies:')
    for name, acc in checkpoint_rows:
        print(f'{name}: {acc:.6f}')
    best_name, best_acc = max(checkpoint_rows, key=lambda x: x[1])
    print(f'\nBest checkpoint: {best_name}')
    print(f'Best validation accuracy: {best_acc:.6f}')


In [ ]:
import subprocess
from pathlib import Path

project_root = Path.cwd()
predict_script = project_root / 'predict_external.py'
best_checkpoint = 'resnet50_bone_cancer_augmented_20ep_20bs.pt'
external_dir = r'D:\BTXRD_PROCESSED\BTXRD_processed\images'
output_file = 'btxrd_predictions_best.csv'

command = [
    str(project_root / '.venv' / 'Scripts' / 'python.exe'),
    str(predict_script),
    '--data-dir', external_dir,
    '--checkpoint', best_checkpoint,
    '--output', output_file,
]
print('Running external prediction...')
subprocess.run(command, cwd=str(project_root), check=True)
print('External prediction complete.')
print(f'Output saved to: {output_file}')
